In [1]:
import time
import math
import torch

from utils.affine_transforms_old import AffineTransformation2D  # full version enums
from utils.affine_transforms import AffineTransformation2D as AffineTransformation2D_SIMPLE  # simple enums

from utils.transform_sequence_old import TransformSequence
from utils.transform_sequence import TransformSequence as TransformSequence2

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float32
print(f"Device: {device}")

Device: cuda


In [2]:
batch_size = 128
tolerance = 1e-5
n_compare_batches = 20         # number of random batches to compare
n_speed_batches = 1000          # for timing (increase for stabler stats)
init_methods = ["individual", "uniform", "latin_hypercube", "sobol"]  # tested init strategies

In [3]:
transforms_full = [
    AffineTransformation2D.ROTATION.value,
    AffineTransformation2D.TRANSLATION.value,
    AffineTransformation2D.SCALING_UNIFORM.value,
]

transforms_simple = [
    AffineTransformation2D_SIMPLE.ROTATION.value,
    AffineTransformation2D_SIMPLE.TRANSLATION.value,
    AffineTransformation2D_SIMPLE.SCALING_UNIFORM.value,
]
domains = [
    (-torch.pi, torch.pi),
    ((-1.0, 1.0), (-2.0, 2.0)),
    (0.5, 1.5),
]

In [4]:
def build_sequences(init_method):
    seq_full = TransformSequence(
        transformations=transforms_full,
        domains=domains,
        device=device,
        dtype=dtype,
        init_method=init_method,
        use_individual_param_correction=True,  # per-transform projection
        reflect=False,
    )
    seq_simple = TransformSequence2(
        transformations=transforms_simple,
        domains=domains,
        device=device,
        dtype=dtype,
        init_method=init_method,
        reflect=False,
    )
    return seq_full, seq_simple

In [5]:
def compare_sequences(seq_full, seq_simple, n_batches, batch_size, tol):
    max_abs = 0.0
    mean_abs_accum = 0.0
    count = 0
    with torch.no_grad():
        for _ in range(n_batches):
            # sample via full version to ensure domain projection
            params = seq_full.sample_individual(batch_size, use_fallback_correction=False)
            # Both sequences should accept same param vector
            A_full = seq_full(params)
            A_simple = seq_simple(params)
            diff = (A_full - A_simple).abs()
            max_abs = max(max_abs, diff.max().item())
            mean_abs_accum += diff.mean().item()
            count += 1
    mean_abs = mean_abs_accum / count
    print(f"Compare: max_abs={max_abs:.3e} mean_abs={mean_abs:.3e} tol={tol}")
    if max_abs > tol:
        print("WARNING: Difference exceeds tolerance.")
    return max_abs, mean_abs

In [6]:
def benchmark_forward(seq, batch_size, n_batches):
    # Pre-sample all params first to isolate forward speed
    params = []
    with torch.no_grad():
        for _ in range(n_batches):
            p = seq.sample_individual(batch_size)
            params.append(p)
    start = time.time()
    with torch.no_grad():
        for p in params:
            _ = seq(p)
    elapsed = time.time() - start
    matrices_per_sec = (n_batches * batch_size) / elapsed
    batches_per_sec = n_batches / elapsed
    return elapsed, matrices_per_sec, batches_per_sec


In [7]:
def benchmark_sampling_plus_forward(seq, batch_size, n_batches):
    start = time.time()
    with torch.no_grad():
        for _ in range(n_batches):
            p = seq.initial_param(batch_size)
            _ = seq(p)
    elapsed = time.time() - start
    matrices_per_sec = (n_batches * batch_size) / elapsed
    batches_per_sec = n_batches / elapsed
    return elapsed, matrices_per_sec, batches_per_sec

In [8]:
seq_full, seq_simple = build_sequences("sobol")

In [9]:
max_abs = 0.0
mean_abs_accum = 0.0
count = 0
with torch.no_grad():
        for _ in range(1):
            # sample via full version to ensure domain projection
            params = seq_full.sample_individual(batch_size, use_fallback_correction=False)
            # Both sequences should accept same param vector
            A_full = seq_full(params)
            A_simple = seq_simple(params)
            diff = (A_full - A_simple).abs()
            max_abs = max(max_abs, diff.max().item())
            mean_abs_accum += diff.mean().item()
            count += 1



In [10]:
results = []
for method in init_methods:
    print(f"\n=== Init method: {method} ===")
    seq_full, seq_simple = build_sequences(method)

    # Equivalence test
    max_abs, mean_abs = compare_sequences(seq_full, seq_simple, n_compare_batches, batch_size, tolerance)

    # Forward-only speed
    e_simple, mps_simple, bps_simple = benchmark_forward(seq_simple, batch_size, n_speed_batches)
    e_full, mps_full, bps_full = benchmark_forward(seq_full, batch_size, n_speed_batches)
    # End-to-end (sampling + forward)
    es_simple, mps_simple_e2e, bps_simple_e2e = benchmark_sampling_plus_forward(seq_simple, batch_size, n_speed_batches)
    es_full, mps_full_e2e, bps_full_e2e = benchmark_sampling_plus_forward(seq_full, batch_size, n_speed_batches)

    print(f"Full forward: {mps_full:.1f} mats/s ({bps_full:.1f} batches/s)")
    print(f"Simpl forward: {mps_simple:.1f} mats/s ({bps_simple:.1f} batches/s)")
    print(f"Full end2end: {mps_full_e2e:.1f} mats/s ({bps_full_e2e:.1f} batches/s)")
    print(f"Simpl end2end: {mps_simple_e2e:.1f} mats/s ({bps_simple_e2e:.1f} batches/s)")

    results.append({
        "init": method,
        "max_abs": max_abs,
        "mean_abs": mean_abs,
        "forward_full_mats_per_s": mps_full,
        "forward_simple_mats_per_s": mps_simple,
        "e2e_full_mats_per_s": mps_full_e2e,
        "e2e_simple_mats_per_s": mps_simple_e2e,
    })


=== Init method: individual ===
Compare: max_abs=0.000e+00 mean_abs=0.000e+00 tol=1e-05
Full forward: 321104.6 mats/s (2508.6 batches/s)
Simpl forward: 355029.7 mats/s (2773.7 batches/s)
Full end2end: 76937.7 mats/s (601.1 batches/s)
Simpl end2end: 70575.7 mats/s (551.4 batches/s)

=== Init method: uniform ===
Compare: max_abs=0.000e+00 mean_abs=0.000e+00 tol=1e-05
Full forward: 363235.1 mats/s (2837.8 batches/s)
Simpl forward: 347294.4 mats/s (2713.2 batches/s)
Full end2end: 316282.7 mats/s (2471.0 batches/s)
Simpl end2end: 211360.9 mats/s (1651.3 batches/s)

=== Init method: latin_hypercube ===
Compare: max_abs=0.000e+00 mean_abs=0.000e+00 tol=1e-05
Full forward: 366112.7 mats/s (2860.3 batches/s)
Simpl forward: 353015.7 mats/s (2757.9 batches/s)
Full end2end: 287020.8 mats/s (2242.3 batches/s)
Simpl end2end: 198025.3 mats/s (1547.1 batches/s)

=== Init method: sobol ===
Compare: max_abs=0.000e+00 mean_abs=0.000e+00 tol=1e-05
Full forward: 369904.0 mats/s (2889.9 batches/s)
Simpl fo

In [11]:
print("\n=== Summary ===")
target = 1000 * batch_size  # interpretation: 1000 batches/sec * batch_size matrices/sec
for r in results:
    print(
        f"{r['init']}: max_abs={r['max_abs']:.2e} "
        f"forward full={r['forward_full_mats_per_s']:.0f}/s "
        f"simpl={r['forward_simple_mats_per_s']:.0f}/s "
        f"e2e full={r['e2e_full_mats_per_s']:.0f}/s "
        f"simpl={r['e2e_simple_mats_per_s']:.0f}/s"
    )


=== Summary ===
individual: max_abs=0.00e+00 forward full=321105/s simpl=355030/s e2e full=76938/s simpl=70576/s
uniform: max_abs=0.00e+00 forward full=363235/s simpl=347294/s e2e full=316283/s simpl=211361/s
latin_hypercube: max_abs=0.00e+00 forward full=366113/s simpl=353016/s e2e full=287021/s simpl=198025/s
sobol: max_abs=0.00e+00 forward full=369904/s simpl=337417/s e2e full=303934/s simpl=204123/s


In [12]:
with torch.no_grad():
    seq_full, seq_simple = build_sequences("individual")
    p = seq_full.sample_individual(1)
    A_full = seq_full(p)
    A_simple = seq_simple(p)
    print("\nSingle sample diff max:", (A_full - A_simple).abs().max().item())
    print("Full matrix:\n", A_full[0])
    print("Simple matrix:\n", A_simple[0])


Single sample diff max: 0.0
Full matrix:
 tensor([[-0.0792, -2.2797,  2.1213],
        [ 2.2797, -0.0792,  1.4962],
        [ 0.0000,  0.0000,  1.0000]], device='cuda:0')
Simple matrix:
 tensor([[-0.0792, -2.2797,  2.1213],
        [ 2.2797, -0.0792,  1.4962],
        [ 0.0000,  0.0000,  1.0000]], device='cuda:0')
